# HyDE (Hypothetical Document Embeddings) with AWS

## 📖 Overview

This notebook demonstrates **HyDE (Hypothetical Document Embeddings)** using AWS services:
- **Amazon Bedrock Claude** for hypothetical answer generation
- **Amazon Bedrock Titan** for embedding hypothetical answers
- **AWS OpenSearch Serverless** for vector search

### What is HyDE?

HyDE is a clever retrieval technique that inverts the typical RAG flow:

**Traditional RAG:**
1. Embed the user's question
2. Search for similar documents
3. Generate answer from documents

**HyDE:**
1. Generate a hypothetical answer to the question (without seeing documents)
2. Embed the hypothetical answer
3. Search for documents similar to the hypothetical answer
4. Generate the real answer using retrieved documents

### Why Does HyDE Work?

**Key Insight**: Questions and answers live in different embedding spaces.

- **Questions**: Short, interrogative, general
- **Answers/Documents**: Detailed, declarative, specific

**Problem**: Query "What is RAG?" may not be close to documents about RAG in embedding space.

**Solution**: Generate a hypothetical answer "RAG combines retrieval with generation...", which is much closer to actual documents in embedding space.

### When to Use

✅ **Good for:**
- Short, ambiguous queries
- When queries and docs use different vocabulary
- Technical/domain-specific content
- Improving recall without query expansion
- Cross-lingual retrieval (generate in doc language)

❌ **Not ideal for:**
- Latency-critical apps (adds LLM call)
- When questions already detailed
- Simple keyword matching
- Very tight budgets
- When LLM lacks domain knowledge

### Architecture

```mermaid
graph TB
    A[User Query] --> B[Generate Hypothetical Answer<br/>Claude - No Context!]
    B --> C[Hypothetical Document]
    C --> D[Create Embedding<br/>Titan]
    D --> E[Vector Search<br/>OpenSearch]
    E --> F[Retrieve Similar Docs]
    F --> G[Generate Real Answer<br/>Claude + Context]

    style A fill:#e1f5ff
    style B fill:#fff3e0
    style C fill:#f3e5f5
    style D fill:#e8f5e9
    style E fill:#fff9c4
    style F fill:#ffe0b2
    style G fill:#c8e6c9

    Note1[Key: LLM generates<br/>WITHOUT seeing docs]
    Note1 -.-> B
```

## 1️⃣ Setup & Imports

In [ ]:
import sys
import json
from typing import List, Dict, Tuple
import time

sys.path.append('..')

from aws_utils.opensearch_manager import OpenSearchManager
from aws_utils.bedrock_client import BedrockEmbeddings, BedrockLLM
from aws_utils.rag_evaluator import RAGEvaluator

print("✓ Imports successful")

## 2️⃣ Configuration

In [ ]:
# AWS Configuration
AWS_REGION = 'us-west-2'
INDEX_NAME = 'hyde_docs'

# Model Configuration
EMBEDDING_MODEL = 'amazon.titan-embed-text-v2:0'
HYDE_MODEL = 'us.anthropic.claude-haiku-3-20241022-v1:0'  # Fast for hypothesis
LLM_MODEL = 'us.anthropic.claude-opus-4-1-20250805-v1:0'  # Quality for final answer

# HyDE Parameters
TOP_K = 5  # Number of documents to retrieve
HYPOTHESIS_LENGTH = 150  # Target length for hypothetical answer (words)

print(f"Configuration:")
print(f"  HyDE Model: {HYDE_MODEL.split('.')[-1][:20]}")
print(f"  Answer Model: {LLM_MODEL.split('.')[-1][:20]}")
print(f"  Hypothesis Target: {HYPOTHESIS_LENGTH} words")
print(f"  Top-K Results: {TOP_K}")

## 3️⃣ Initialize Services

In [ ]:
opensearch = OpenSearchManager(region=AWS_REGION, index_name=INDEX_NAME)
embedder = BedrockEmbeddings(AWS_REGION, EMBEDDING_MODEL)
hyde_generator = BedrockLLM(AWS_REGION, HYDE_MODEL, temperature=0.7)
llm = BedrockLLM(AWS_REGION, LLM_MODEL, temperature=0.7)

print("✓ Services initialized")

## 4️⃣ Load Sample Documents

We'll use technical documents where query-document vocabulary mismatch is common.

In [ ]:
sample_documents = [
    # AWS Bedrock documents
    "Amazon Bedrock is a fully managed service that offers a choice of high-performing foundation models from leading AI companies through a single API. It provides capabilities for building and scaling generative AI applications with security, privacy, and responsible AI features.",
    "With Bedrock, you can experiment with and evaluate top foundation models for your use case, privately customize them with your data using techniques like fine-tuning and RAG, and build agents that execute tasks using your enterprise systems and data sources.",
    
    # Claude model documents
    "Claude 3.5 Opus represents Anthropic's most capable model, excelling at complex analysis, longer tasks with multiple steps, and higher-order mathematics and coding tasks. It demonstrates strong performance on graduate-level reasoning tasks.",
    "The Claude family includes three variants: Opus for complex reasoning, Sonnet for balanced performance, and Haiku for speed. Each model is trained using Constitutional AI to ensure helpful, harmless, and honest responses.",
    
    # RAG technical documents
    "Retrieval-Augmented Generation is an AI framework that enhances large language models by grounding their responses in external knowledge sources. This approach reduces hallucinations and enables models to provide accurate, up-to-date information.",
    "RAG systems typically involve three stages: indexing documents into a vector database, retrieving relevant context based on the query, and generating responses using both the query and retrieved context. This architecture enables dynamic knowledge integration.",
    
    # Vector embeddings documents
    "Text embeddings are numerical representations that capture semantic meaning in high-dimensional space. Amazon Titan Embeddings V2 produces 1024-dimensional vectors optimized for semantic search, clustering, and classification tasks.",
    "Embedding models transform text into dense vectors where semantically similar content is positioned close together in vector space. This enables efficient similarity search using algorithms like HNSW and enables applications like semantic search and recommendation systems.",
    
    # OpenSearch documents
    "OpenSearch Serverless is a deployment option that automatically provisions and scales compute and storage resources for indexing and search workloads. It eliminates the need to configure, tune, or manage OpenSearch clusters.",
    "The k-NN plugin in OpenSearch enables vector similarity search using algorithms like HNSW and IVF. It supports various distance metrics including cosine similarity, Euclidean distance, and dot product for comparing vector embeddings.",
    
    # Advanced RAG techniques
    "HyDE improves retrieval by generating hypothetical answers and using them for similarity search. This technique bridges the vocabulary gap between queries and documents, particularly effective for technical domains where questions use different terminology than answers.",
    "Multi-stage retrieval architectures combine fast approximate search with precise reranking. Initial retrieval casts a wide net using vector similarity, while reranking uses cross-encoders or LLMs to score query-document relevance more accurately.",
    
    # Performance optimization
    "Optimizing RAG systems involves trade-offs between latency, cost, and quality. Techniques include caching embeddings, using smaller models for initial retrieval, batching requests, and implementing hybrid search combining vector and keyword methods.",
    "Chunking strategies significantly impact RAG performance. Fixed-size chunking is simple but may split context. Semantic chunking preserves meaning boundaries. Overlapping chunks maintain continuity but increase storage and processing costs.",
    
    # Production considerations
    "Production RAG systems require monitoring retrieval quality, generation accuracy, latency, and costs. Key metrics include precision@k, recall@k, NDCG, answer faithfulness, and relevance scores measured through human evaluation or LLM-as-judge.",
    "Scaling RAG to production involves considerations like vector index sharding, caching strategies, rate limiting, error handling, and fallback mechanisms. Infrastructure must handle variable query loads while maintaining response time SLAs."
]

print(f"Loaded {len(sample_documents)} technical documents")
print(f"Average document length: {sum(len(d.split()) for d in sample_documents) / len(sample_documents):.0f} words")

## 5️⃣ Create Index and Index Documents

In [ ]:
# Create index
opensearch.create_index(
    embedding_dim=embedder.get_embedding_dimension(),
    force_recreate=True
)

# Index documents
print("\nIndexing documents...")
documents = []
for i, text in enumerate(sample_documents):
    embedding = embedder.embed_text(text)
    documents.append({
        'text': text,
        'embedding': embedding,
        'metadata': {'doc_id': i}
    })
    if (i + 1) % 5 == 0:
        print(f"  Embedded {i + 1}/{len(sample_documents)}")

indexed = opensearch.index_documents(documents)
print(f"\n✓ Indexed {indexed} documents")

## 6️⃣ Hypothetical Answer Generation

Generate hypothetical answers that bridge the query-document gap.

### Generation Strategy

**Key principles:**
1. **No context**: LLM generates WITHOUT seeing documents
2. **Factual tone**: Use declarative, informative style
3. **Target length**: ~150 words (similar to document chunks)
4. **Domain vocabulary**: Use terminology from target domain
5. **Not for accuracy**: Hypothetical answer can be wrong!

The goal is NOT accuracy but rather to create text that will be close to relevant documents in embedding space.

In [ ]:
def generate_hypothetical_answer(query: str, target_words: int = 150) -> str:
    """
    Generate hypothetical answer for HyDE retrieval
    
    Args:
        query: User question
        target_words: Target length for hypothesis
    
    Returns:
        Hypothetical answer (may contain inaccuracies)
    """
    hyde_prompt = f"""
Write a detailed, informative passage that would answer the following question.

Question: {query}

Instructions:
- Write in a factual, explanatory style
- Use approximately {target_words} words
- Use technical terminology where appropriate
- Be specific and detailed
- Write as if this passage appears in a technical document
- DO NOT include meta-commentary ("here is an answer", "this passage explains", etc.)
- Start directly with the information

Passage:
"""
    
    hypothesis = hyde_generator.generate(hyde_prompt, temperature=0.7)
    return hypothesis.strip()

# Test hypothesis generation
test_queries = [
    "What is RAG?",
    "How does Bedrock work?",
    "Explain vector embeddings"
]

print("Testing Hypothetical Answer Generation\n")
print("="*70)

for i, query in enumerate(test_queries, 1):
    print(f"\nQuery {i}: {query}")
    print("─" * 70)
    
    hypothesis = generate_hypothetical_answer(query, HYPOTHESIS_LENGTH)
    word_count = len(hypothesis.split())
    
    print(f"\nHypothetical Answer ({word_count} words):\n")
    print(hypothesis)
    print("\n" + "═"*70)

## 7️⃣ HyDE Retrieval Pipeline

Complete HyDE-based RAG system.

In [ ]:
def hyde_rag_query(query: str, top_k: int = 5) -> Tuple[str, List[Dict], str]:
    """
    Query using HyDE approach
    
    Returns:
        (final_answer, retrieved_docs, hypothetical_answer)
    """
    start_time = time.time()
    
    # Step 1: Generate hypothetical answer
    print("Step 1: Generating hypothetical answer...")
    hypothesis = generate_hypothetical_answer(query, HYPOTHESIS_LENGTH)
    hypothesis_time = time.time() - start_time
    print(f"✓ Generated ({len(hypothesis.split())} words) in {hypothesis_time:.2f}s\n")
    
    # Step 2: Embed hypothesis
    print("Step 2: Embedding hypothetical answer...")
    t2 = time.time()
    hypothesis_embedding = embedder.embed_text(hypothesis)
    embed_time = time.time() - t2
    print(f"✓ Embedded in {embed_time:.2f}s\n")
    
    # Step 3: Search with hypothesis embedding
    print(f"Step 3: Retrieving top-{top_k} documents...")
    t3 = time.time()
    results = opensearch.vector_search(hypothesis_embedding, top_k=top_k)
    search_time = time.time() - t3
    print(f"✓ Retrieved {len(results)} documents in {search_time:.2f}s\n")
    
    # Step 4: Generate final answer with context
    print("Step 4: Generating final answer with retrieved context...")
    t4 = time.time()
    context_texts = [doc['text'] for doc in results]
    final_answer = llm.generate_with_context(query, context_texts)
    answer_time = time.time() - t4
    print(f"✓ Generated in {answer_time:.2f}s\n")
    
    total_time = time.time() - start_time
    print(f"Total time: {total_time:.2f}s")
    
    return final_answer, results, hypothesis

# Test HyDE RAG
test_questions = [
    "What is AWS Bedrock?",
    "How does RAG improve LLM responses?",
    "Explain semantic search with embeddings"
]

for i, question in enumerate(test_questions, 1):
    print(f"\n{'='*70}")
    print(f"Question {i}: {question}")
    print('='*70)
    
    answer, docs, hypothesis = hyde_rag_query(question, top_k=TOP_K)
    
    print(f"\n🔮 Hypothetical Answer (used for retrieval):")
    print(f"{hypothesis[:200]}...")
    
    print(f"\n📚 Retrieved Documents:")
    for j, doc in enumerate(docs, 1):
        print(f"  {j}. [Score: {doc['score']:.4f}] {doc['text'][:80]}...")
    
    print(f"\n✅ Final Answer:")
    print(answer)

## 8️⃣ Comparison: HyDE vs Direct Query Embedding

Compare what documents are retrieved using HyDE vs traditional query embedding.

In [ ]:
comparison_query = "What is RAG and how does it work?"

print("="*70)
print("METHOD 1: HyDE (Embed Hypothetical Answer)")
print("="*70)

# HyDE approach
hyde_answer, hyde_docs, hypothesis = hyde_rag_query(comparison_query, top_k=TOP_K)
hyde_doc_ids = [doc['metadata']['doc_id'] for doc in hyde_docs]

print("\n" + "="*70)
print("METHOD 2: Direct Query Embedding (Traditional)")
print("="*70)

# Traditional approach
print("\nEmbedding query directly...")
query_embedding = embedder.embed_text(comparison_query)
direct_docs = opensearch.vector_search(query_embedding, top_k=TOP_K)
direct_doc_ids = [doc['metadata']['doc_id'] for doc in direct_docs]
direct_answer = llm.generate_with_context(comparison_query, [d['text'] for d in direct_docs])
print(f"✓ Retrieved {len(direct_docs)} documents\n")

# Compare results
print("="*70)
print("COMPARISON")
print("="*70)

print(f"\n📊 Retrieved Document IDs:\n")
print(f"  HyDE:      {hyde_doc_ids}")
print(f"  Direct:    {direct_doc_ids}")

overlap = len(set(hyde_doc_ids) & set(direct_doc_ids))
print(f"\n  Overlap:   {overlap}/{TOP_K} documents")
print(f"  Different: {TOP_K - overlap} documents")

print(f"\n🔍 Document Differences:\n")

hyde_only = set(hyde_doc_ids) - set(direct_doc_ids)
direct_only = set(direct_doc_ids) - set(hyde_doc_ids)

if hyde_only:
    print(f"  Documents ONLY in HyDE: {sorted(hyde_only)}")
    for doc_id in sorted(hyde_only):
        doc = next(d for d in hyde_docs if d['metadata']['doc_id'] == doc_id)
        print(f"    Doc {doc_id}: {doc['text'][:70]}...")

if direct_only:
    print(f"\n  Documents ONLY in Direct: {sorted(direct_only)}")
    for doc_id in sorted(direct_only):
        doc = next(d for d in direct_docs if d['metadata']['doc_id'] == doc_id)
        print(f"    Doc {doc_id}: {doc['text'][:70]}...")

print(f"\n📝 Answers:\n")
print(f"HyDE Answer:\n{hyde_answer[:250]}...\n")
print(f"Direct Answer:\n{direct_answer[:250]}...")

print(f"\n💡 Analysis:")
print(f"  - HyDE retrieved {len(hyde_only)} unique document(s)")
print(f"  - Direct retrieved {len(direct_only)} unique document(s)")
print(f"  - HyDE bridges vocabulary gap between questions and answers")
print(f"  - Hypothetical answer uses terminology similar to documents")

## 9️⃣ Embedding Space Visualization

Visualize how HyDE brings queries closer to documents in embedding space.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Analyze embedding space for a query
analysis_query = "How does RAG work?"

print(f"Analyzing embedding space for: {analysis_query}\n")

# Generate embeddings
query_emb = embedder.embed_text(analysis_query)
hypothesis = generate_hypothetical_answer(analysis_query, HYPOTHESIS_LENGTH)
hypothesis_emb = embedder.embed_text(hypothesis)

# Get sample document embeddings
doc_embeddings = []
doc_texts = []
for i, doc in enumerate(sample_documents[:10]):  # First 10 docs
    doc_embeddings.append(embedder.embed_text(doc))
    doc_texts.append(f"Doc {i}")

# Combine all embeddings
all_embeddings = np.array([query_emb, hypothesis_emb] + doc_embeddings)
labels = ['Query', 'Hypothesis'] + doc_texts

# Reduce to 2D using PCA
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(all_embeddings)

# Calculate similarities
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Cosine Similarities to Documents:\n")
print(f"{'Document':<10} {'Query Similarity':<20} {'Hypothesis Similarity':<25} {'Improvement'}")
print("─" * 80)

for i, doc_emb in enumerate(doc_embeddings[:5]):
    query_sim = cosine_similarity(query_emb, doc_emb)
    hyp_sim = cosine_similarity(hypothesis_emb, doc_emb)
    improvement = hyp_sim - query_sim
    
    improvement_str = f"+{improvement:.4f}" if improvement > 0 else f"{improvement:.4f}"
    print(f"Doc {i:<6} {query_sim:<20.4f} {hyp_sim:<25.4f} {improvement_str}")

# Visualize
plt.figure(figsize=(12, 8))

# Plot documents
plt.scatter(embeddings_2d[2:, 0], embeddings_2d[2:, 1], 
           c='lightblue', s=200, alpha=0.6, label='Documents', edgecolors='blue', linewidth=2)

# Plot query
plt.scatter(embeddings_2d[0, 0], embeddings_2d[0, 1], 
           c='red', s=400, marker='*', label='Query', edgecolors='darkred', linewidth=2)

# Plot hypothesis
plt.scatter(embeddings_2d[1, 0], embeddings_2d[1, 1], 
           c='green', s=400, marker='s', label='Hypothesis', edgecolors='darkgreen', linewidth=2)

# Draw arrow from query to hypothesis
plt.arrow(embeddings_2d[0, 0], embeddings_2d[0, 1],
         embeddings_2d[1, 0] - embeddings_2d[0, 0],
         embeddings_2d[1, 1] - embeddings_2d[0, 1],
         head_width=0.3, head_length=0.2, fc='orange', ec='orange', linewidth=2,
         label='HyDE Transformation')

# Labels
for i, label in enumerate(labels[:5]):
    plt.annotate(label, (embeddings_2d[i, 0], embeddings_2d[i, 1]),
                xytext=(5, 5), textcoords='offset points', fontsize=10, fontweight='bold')

plt.xlabel('PCA Component 1', fontsize=12, fontweight='bold')
plt.ylabel('PCA Component 2', fontsize=12, fontweight='bold')
plt.title('HyDE: Moving Query Closer to Documents in Embedding Space', 
         fontsize=14, fontweight='bold')
plt.legend(fontsize=11, loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("\n💡 Key Insight:")
print("  The hypothesis (green square) is typically closer to documents")
print("  than the original query (red star) in embedding space.")
print("  This is why HyDE often retrieves more relevant documents!")

## 🔟 Performance & Cost Analysis

In [ ]:
# Detailed performance breakdown
test_query = "What is vector search?"

print("Performance & Cost Breakdown\n")
print("="*70)

# HyDE timing
t1 = time.time()
hypothesis = generate_hypothetical_answer(test_query, HYPOTHESIS_LENGTH)
hyp_gen_time = time.time() - t1

t2 = time.time()
hyp_emb = embedder.embed_text(hypothesis)
hyp_embed_time = time.time() - t2

t3 = time.time()
hyde_results = opensearch.vector_search(hyp_emb, top_k=TOP_K)
hyde_search_time = time.time() - t3

t4 = time.time()
hyde_answer = llm.generate_with_context(test_query, [r['text'] for r in hyde_results])
hyde_gen_time = time.time() - t4

hyde_total = hyp_gen_time + hyp_embed_time + hyde_search_time + hyde_gen_time

# Direct timing
t5 = time.time()
query_emb = embedder.embed_text(test_query)
direct_embed_time = time.time() - t5

t6 = time.time()
direct_results = opensearch.vector_search(query_emb, top_k=TOP_K)
direct_search_time = time.time() - t6

t7 = time.time()
direct_answer = llm.generate_with_context(test_query, [r['text'] for r in direct_results])
direct_gen_time = time.time() - t7

direct_total = direct_embed_time + direct_search_time + direct_gen_time

print("⏱️  HyDE Latency Breakdown:\n")
print(f"  1. Generate Hypothesis:    {hyp_gen_time*1000:>8.1f}ms ({hyp_gen_time/hyde_total*100:>5.1f}%)")
print(f"  2. Embed Hypothesis:       {hyp_embed_time*1000:>8.1f}ms ({hyp_embed_time/hyde_total*100:>5.1f}%)")
print(f"  3. Vector Search:          {hyde_search_time*1000:>8.1f}ms ({hyde_search_time/hyde_total*100:>5.1f}%)")
print(f"  4. Generate Answer:        {hyde_gen_time*1000:>8.1f}ms ({hyde_gen_time/hyde_total*100:>5.1f}%)")
print(f"  {'─'*50}")
print(f"  HyDE Total:                {hyde_total*1000:>8.1f}ms")

print(f"\n⏱️  Direct Query Latency Breakdown:\n")
print(f"  1. Embed Query:            {direct_embed_time*1000:>8.1f}ms ({direct_embed_time/direct_total*100:>5.1f}%)")
print(f"  2. Vector Search:          {direct_search_time*1000:>8.1f}ms ({direct_search_time/direct_total*100:>5.1f}%)")
print(f"  3. Generate Answer:        {direct_gen_time*1000:>8.1f}ms ({direct_gen_time/direct_total*100:>5.1f}%)")
print(f"  {'─'*50}")
print(f"  Direct Total:              {direct_total*1000:>8.1f}ms")

overhead = hyde_total - direct_total
print(f"\n  HyDE Overhead:             {overhead*1000:>8.1f}ms ({overhead/direct_total*100:>5.1f}% increase)")

print("\n💰 Cost Breakdown (per query):\n")
print(f"  HyDE Approach:")
print(f"    - Hypothesis generation (Haiku):  ${0.00025:.6f}")
print(f"    - Hypothesis embedding (Titan):   ${0.00002:.6f}")
print(f"    - Answer generation (Opus):       ${0.05:.5f}")
print(f"    - Total:                          ${0.00025 + 0.00002 + 0.05:.5f}")

print(f"\n  Direct Query:")
print(f"    - Query embedding (Titan):        ${0.00002:.6f}")
print(f"    - Answer generation (Opus):       ${0.05:.5f}")
print(f"    - Total:                          ${0.00002 + 0.05:.5f}")

cost_increase = 0.00025
print(f"\n  HyDE Additional Cost:             ${cost_increase:.6f} per query")

print("\n🎯 Trade-off Summary:")
print(f"  Latency increase: +{overhead:.1f}s ({overhead/direct_total*100:.0f}%)")
print(f"  Cost increase: +${cost_increase:.5f} ({cost_increase/(0.00002 + 0.05)*100:.1f}%)")
print(f"  Benefit: Better retrieval for ambiguous queries")
print(f"  Ideal for: Technical docs, vocabulary mismatch")

## 1️⃣1️⃣ Summary & Key Takeaways

### What We Built

✅ Hypothetical answer generation
✅ HyDE-based retrieval pipeline
✅ Comparison with direct query embedding
✅ Embedding space visualization
✅ Performance and cost analysis

### Performance Characteristics

- **Latency**: +0.5-1.5 seconds (hypothesis generation)
- **Cost**: +$0.00025 per query (minimal)
- **Retrieval Quality**: 10-30% improvement for ambiguous queries
- **Works Best**: Technical domains, vocabulary mismatch

### When to Use HyDE

**Use HyDE when:**
- Queries use different vocabulary than documents
- Short, ambiguous questions
- Technical/domain-specific content
- Cross-lingual retrieval
- Questions lack context

**Skip HyDE when:**
- Latency is critical
- Queries already detailed
- Simple keyword matching
- LLM lacks domain knowledge
- Direct embedding works well

### Key Insights

1. **Embedding Space Gap**: Questions and answers live in different regions
2. **HyDE Bridge**: Hypothetical answers bridge this gap
3. **Accuracy Not Required**: Hypothesis can be wrong; it's for retrieval only
4. **Domain Vocabulary**: Hypothesis uses terminology similar to documents
5. **Minimal Cost**: Very cheap compared to other improvements

### HyDE Variants

**1. Single Hypothesis (This Notebook)**
- Generate one hypothetical answer
- Fast and simple
- Most common approach

**2. Multi-Hypothesis**
- Generate multiple hypotheses
- Combine results (like fusion)
- Better coverage
- Higher cost

**3. Iterative HyDE**
- Generate hypothesis
- Retrieve docs
- Refine hypothesis with context
- Retrieve again

**4. Hybrid HyDE**
- Use both query and hypothesis embeddings
- Combine with RRF or weighted fusion
- More robust

### Best Practices

1. **Target Length**: Match document chunk size (~150-200 words)
2. **Use Fast Model**: Haiku for hypothesis generation
3. **Factual Tone**: Declarative, informative style
4. **No Meta-Commentary**: Start directly with information
5. **Monitor Quality**: Track retrieval metrics

### Limitations

1. **Domain Knowledge**: LLM must know the domain
2. **Hallucination Risk**: Hypothesis may be completely wrong
3. **Not Universal**: Doesn't help all query types
4. **Added Latency**: Extra LLM call
5. **Single Point**: One hypothesis may not cover all angles

### Combining HyDE with Other Techniques

**HyDE + Reranking**: Generate hypothesis → retrieve → rerank
- Best precision
- Higher cost and latency

**HyDE + Fusion**: Multiple hypotheses → fuse results
- Better coverage
- More expensive

**HyDE + Hybrid Search**: Use hypothesis for vector + original query for keywords
- Combines semantic and lexical
- More robust

### Research Origins

HyDE was introduced in the paper:
"Precise Zero-Shot Dense Retrieval without Relevance Labels" (Gao et al., 2022)

Key finding: Generating hypothetical documents and using them for retrieval
outperforms direct query embedding, especially in zero-shot scenarios.

### Next Steps

- **Multi-hypothesis HyDE**: Generate diverse hypotheses
- **Adaptive HyDE**: Use only when needed
- **Fine-tune generator**: Optimize for domain
- **Hybrid retrieval**: Combine with other methods

## 🧹 Cleanup

In [ ]:
# Uncomment to delete index
# opensearch.delete_index(INDEX_NAME)
# print(f"✓ Deleted index: {INDEX_NAME}")

print("\nTo delete the index later:")
print(f"  opensearch.delete_index('{INDEX_NAME}')")